In [ ]:
# validate the downscaled data using the same framework with all raw datasets

# V1 - 2026-07-29 - create code - jpk

In [26]:
# libraries
import pandas as pd
import numpy as np
import glob
import os
import rasterio
import HydroErr as he
import concurrent.futures
from pyproj import Transformer

#paths
insitu_tmax_raw = pd.read_parquet(r"F:\ClimateDataWS\Data\Insitu\Cleaned\tmax_cleaned.parquet")
insitu_tmin_raw = pd.read_parquet(r"F:\ClimateDataWS\Data\Insitu\Cleaned\tmin_cleaned.parquet")

tmax_train_df = r"F:\ClimateDataWS\Data\Processed\Split_stations\tmax_train_stations.csv"
tmax_test_df = r"F:\ClimateDataWS\Data\Processed\Split_stations\tmax_test_stations.csv"
tmin_train_df = r"F:\ClimateDataWS\Data\Processed\Split_stations\tmin_train_stations.csv"
tmin_test_df = r"F:\ClimateDataWS\Data\Processed\Split_stations\tmin_test_stations.csv"

output_dir = r"F:\ClimateDataWS\Data\Processed\Daily_bias_correction\Validation_results"

os.makedirs(output_dir, exist_ok=True)

#parameters
datasets = ['ERA5L', 'HAR', 'CHIRTS', 'CHELSA', 'IS_K', 'DS']
start_year, start_month = 2001, 1
end_year,   end_month   = 2025, 9
start_date = pd.Timestamp(f"{start_year}-{start_month:02d}-01")
end_date = pd.Timestamp(f"{end_year}-{end_month:02d}-01") + pd.offsets.MonthEnd(0)

In [27]:
# 1 Read insitu and separate train-test

# process tmax ==
insitu_tmax_raw["date"]  = pd.to_datetime(insitu_tmax_raw[["year", "month", "day"]])
insitu_tmax_raw  = insitu_tmax_raw.rename(columns={"lat": "y", "long": "x", "max_temp": "tmax"})

insitu_tmax_daily = insitu_tmax_raw[
    (insitu_tmax_raw["year"]  >= start_year) &
    (insitu_tmax_raw["year"]  <= end_year)
].copy()
insitu_tmax_daily[["x", "y"]] = insitu_tmax_daily[["x", "y"]].round(5)

# Train/test split
train_ids_tmax = set(pd.read_csv(tmax_train_df)["index_no"])
test_ids_tmax = set(pd.read_csv(tmax_test_df)["index_no"])

insitu_tmax_daily_train = insitu_tmax_daily[insitu_tmax_daily["index_no"].isin(train_ids_tmax)].copy()
insitu_tmax_daily_test = insitu_tmax_daily[insitu_tmax_daily["index_no"].isin(test_ids_tmax)].copy()

# Unique test stations — order must stay fixed (iloc used in sampling)
unique_stn_tmax = (
    insitu_tmax_daily_test[['index_no', 'x', 'y']]
    .drop_duplicates('index_no')
    .reset_index(drop=True)
)
stn_points_tmax = list(zip(unique_stn_tmax['x'], unique_stn_tmax['y']))  


# process tmin ==
insitu_tmin_raw["date"]  = pd.to_datetime(insitu_tmin_raw[["year", "month", "day"]])
insitu_tmin_raw  = insitu_tmin_raw.rename(columns={"lat": "y", "long": "x", "min_temp": "tmin"})

insitu_tmin_daily = insitu_tmin_raw[
    (insitu_tmin_raw["year"]  >= start_year) &
    (insitu_tmin_raw["year"]  <= end_year)
].copy()
insitu_tmin_daily[["x", "y"]] = insitu_tmin_daily[["x", "y"]].round(5)

train_ids_tmin = set(pd.read_csv(tmin_train_df)["index_no"])
test_ids_tmin = set(pd.read_csv(tmin_test_df)["index_no"])

insitu_tmin_daily_train = insitu_tmin_daily[insitu_tmin_daily["index_no"].isin(train_ids_tmin)].copy()
insitu_tmin_daily_test = insitu_tmin_daily[insitu_tmin_daily["index_no"].isin(test_ids_tmin)].copy()

# Unique test stations — order must stay fixed (iloc used in sampling)
unique_stn_tmin = (
    insitu_tmin_daily_test[['index_no', 'x', 'y']]
    .drop_duplicates('index_no')
    .reset_index(drop=True)
)
stn_points_tmin = list(zip(unique_stn_tmin['x'], unique_stn_tmin['y']))  

In [28]:
#Helper

def extract_date(filepath, dataset):
    """Parse date from filename"""
    fname = os.path.basename(filepath)
    return pd.to_datetime(fname[-12:-4], format='%Y%m%d')


def get_dataset_paths(t_variable):
    return {'ERA5L': r"F:\ClimateDataWS\Data\Datasets\ERA5L",
    'HAR': r"F:\ClimateDataWS\Data\Datasets\HAR",
    'CHIRTS':r"F:\ClimateDataWS\Data\Datasets\CHIRTS",
    'CHELSA': r"F:\ClimateDataWS\Data\Datasets\CHELSA",
    'IS_K': fr"F:\ClimateDataWS\Data\Insitu\KrigingDaily_1Km\{t_variable}",
    'DS': fr"F:\ClimateDataWS\Data\Processed\Daily_bias_correction\QRF_products\{t_variable}\Deterministic"
    }

def build_dataset_results(dataset_paths, datasets, t_variable):
    """Build {dataset}_files dict pre-filtered to analysis period."""
    dataset_results = {}
    for dataset in datasets:
        if dataset in ['ERA5L', 'HAR', 'CHIRTS']:
            files = sorted(glob.glob(os.path.join(dataset_paths[dataset], f"{dataset.lower()}_{t_variable}_*.tif")))
        elif dataset == 'CHELSA':
            files = sorted(glob.glob(os.path.join(dataset_paths[dataset], f"{dataset}_{t_variable}_*.tif")))
        else:
            files = sorted(glob.glob(os.path.join(dataset_paths[dataset], "*.tif")))
        filtered = []
        for f in files:
            try:
                d = extract_date(f, dataset)
                if start_date <= d <= end_date:
                    filtered.append(f)
            except Exception as e:
                print(f"  Skipping {os.path.basename(f)}: {e}")
        dataset_results[f'{dataset}_files'] = filtered
        print(f"{dataset}: {len(filtered)} files in period")
    return dataset_results

In [29]:
def _sample_file(file, dataset, stn_points, unique_stn, t_variable):
    date_obj = extract_date(file, dataset)
    try:
        with rasterio.open(file) as src:
            nodata_val = src.nodata
            if src.crs.to_epsg() != 4326:
                transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
                pts = [transformer.transform(x, y) for x, y in stn_points]
            else:
                pts = stn_points
            sampled = list(src.sample(pts))
        return [
            {
                'date': date_obj,
                'stn_id': unique_stn.iloc[i]['index_no'],
                f'{dataset}_{t_variable}': (
                    np.nan if (nodata_val is not None and v[0] == nodata_val) else v[0]
                ),
            }
            for i, v in enumerate(sampled)
        ]
    except Exception as e:
        print(f"  Error processing {os.path.basename(file)}: {e}")
        return []

def extract_daily_datasets(dataset_results, stn_points, unique_stn, datasets, t_variable):
    """
      Sample each file concurrently and merge at daily level
    """
    df_final = None

    for dataset in datasets:
        files = dataset_results.get(f'{dataset}_files', [])
        if not files:
            print(f"Warning: no files found for {dataset}, skipping.")
            continue
        print(f"Processing {dataset} ({len(files)} daily files)...")

        records = []
        with concurrent.futures.ThreadPoolExecutor() as executor:
            for result in executor.map(
                lambda f: _sample_file(f, dataset, stn_points, unique_stn, t_variable), files
            ):
                records.extend(result)

        df_dataset = pd.DataFrame(records)
        if df_dataset.empty:
            print(f"  Warning: no data extracted for {dataset}")
            continue

        df_dataset['date'] = pd.to_datetime(df_dataset['date'])
        df_dataset = df_dataset.assign(
            year  = lambda d: d['date'].dt.year,
            month = lambda d: d['date'].dt.month,
            day = lambda d: d['date'].dt.day
        ).drop(columns='date')

        df_final = (
            df_dataset if df_final is None
            else pd.merge(df_final, df_dataset,
                          on=['stn_id', 'year', 'month', 'day'], how='outer')
        )
        print(f"  → merged shape: {df_final.shape}")

    return df_final

In [30]:
# Build dataset results - Tmax
tmax_results = build_dataset_results(get_dataset_paths("tmax"), datasets, "tmax")

df_grid_tmax = extract_daily_datasets(
    tmax_results, stn_points_tmax, unique_stn_tmax, datasets, "tmax" 
)

df_validation_tmax = pd.merge(
    insitu_tmax_daily_test.rename(columns={'index_no': 'stn_id'}),
    df_grid_tmax,
    on=['stn_id', 'year', 'month', 'day'],
    how='inner'
)
df_validation_tmax.to_csv(os.path.join(output_dir, "Tmax_validation_df.csv"), index = False)

# Build dataset results - Tmin
tmin_results = build_dataset_results(get_dataset_paths("tmin"), datasets, "tmin")

df_grid_tmin = extract_daily_datasets(
    tmin_results, stn_points_tmin, unique_stn_tmin, datasets, "tmin" 
)

df_validation_tmin = pd.merge(
    insitu_tmin_daily_test.rename(columns={'index_no': 'stn_id'}),
    df_grid_tmin,
    on=['stn_id', 'year', 'month', 'day'],
    how='inner'
)
df_validation_tmin.to_csv(os.path.join(output_dir, "Tmin_validation_df.csv"), index = False)

ERA5L: 9039 files in period
HAR: 9039 files in period
CHIRTS: 9039 files in period
CHELSA: 9036 files in period
IS_K: 9039 files in period
DS: 9039 files in period
Processing ERA5L (9039 daily files)...
  → merged shape: (108468, 5)
Processing HAR (9039 daily files)...
  → merged shape: (108468, 6)
Processing CHIRTS (9039 daily files)...
  → merged shape: (108468, 7)
Processing CHELSA (9036 daily files)...
  → merged shape: (108468, 8)
Processing IS_K (9039 daily files)...
  → merged shape: (108468, 9)
Processing DS (9039 daily files)...
  → merged shape: (108468, 10)
ERA5L: 9039 files in period
HAR: 9039 files in period
CHIRTS: 9039 files in period
CHELSA: 9036 files in period
IS_K: 9039 files in period
DS: 9039 files in period
Processing ERA5L (9039 daily files)...
  → merged shape: (108468, 5)
Processing HAR (9039 daily files)...
  → merged shape: (108468, 6)
Processing CHIRTS (9039 daily files)...
  → merged shape: (108468, 7)
Processing CHELSA (9036 daily files)...
  → merged shap

In [31]:
def normalize_column(col):
    if col.max() == col.min():
        return pd.Series(0, index=col.index)
    return (col - col.min()) / (col.max() - col.min())
          
def calculate_accuracy(dataframe_temp, t_variable, datasets_list):
    datasets_metrics = []

    for dataset in datasets_list:
        dataset_accuracy = {'dataset': dataset}

        grid_col = f'{dataset}_{t_variable}'
        insitu_col = f'{t_variable}'
        
        if grid_col not in dataframe_temp.columns:
            print(f"Skipping {dataset} - column not found.")
            continue

        temp_grid_array = dataframe_temp[grid_col].values
        temp_insitu_array = dataframe_temp[insitu_col].values
    
        valid_indices = np.logical_and.reduce([
            ~np.isnan(temp_grid_array),
            ~np.isnan(temp_insitu_array)
        ])
    
        observations = np.sum(valid_indices)
        dataset_accuracy['observations_n'] = observations

        if observations == 0:
            continue

        temp_grid_array = temp_grid_array[valid_indices]
        temp_insitu_array = temp_insitu_array[valid_indices]

        dataset_accuracy['correlation'] = he.spearman_r(temp_grid_array, temp_insitu_array)
        dataset_accuracy['nrmse'] = he.nrmse_mean(temp_grid_array, temp_insitu_array)
        dataset_accuracy['rsd'] = (np.std(temp_grid_array)) / (np.std(temp_insitu_array))

        datasets_metrics.append(dataset_accuracy)

    df_accuracy = pd.DataFrame(datasets_metrics)

    df_accuracy['diff_rsd'] = (df_accuracy['rsd'] - 1).abs()
    
    df_accuracy_normalized = df_accuracy.copy()

    metrics = ['correlation', 'nrmse', 'diff_rsd']
    for column in metrics:
        df_accuracy_normalized[column] = normalize_column(df_accuracy_normalized[column])

    df_accuracy_normalized['correlation'] = 1 - df_accuracy_normalized['correlation']

    df_accuracy_normalized['cpi'] = (df_accuracy_normalized['correlation'] + 
                                     df_accuracy_normalized['nrmse'] + 
                                     df_accuracy_normalized['diff_rsd'])
    df_accuracy_normalized['cpi_rank'] = df_accuracy_normalized['cpi'].rank(ascending=True)

    df_accuracy_final = df_accuracy_normalized[['dataset','cpi','cpi_rank']].merge(
        df_accuracy[['dataset','correlation','nrmse', 'rsd', 'observations_n']], 
        on=['dataset'], 
        how='inner'
    )
    return df_accuracy_final


In [32]:
results_tmax = calculate_accuracy(df_validation_tmax, "tmax", datasets)
results_tmax

,dataset,cpi,cpi_rank,correlation,nrmse,rsd,observations_n
0,ERA5L,2.357741,5.0,0.839967,0.208058,1.191384,105073
1,HAR,3.000000,6.0,0.806033,0.214546,1.294798,105073
2,CHIRTS,0.286710,2.0,0.926206,0.086998,0.979678,105073
3,CHELSA,1.107053,4.0,0.920556,0.148445,1.118812,105037
4,IS_K,0.000000,1.0,0.946795,0.075725,0.996961,105073
5,DS,0.512202,3.0,0.905367,0.093823,0.971427,105073


In [33]:
results_tmin = calculate_accuracy(df_validation_tmin, "tmin", datasets)
results_tmin

,dataset,cpi,cpi_rank,correlation,nrmse,rsd,observations_n
0,ERA5L,2.807035,6.0,0.912088,0.239098,1.146997,103124
1,HAR,2.018836,5.0,0.899711,0.206594,1.071902,103124
2,CHIRTS,0.896935,3.0,0.946462,0.197897,1.037522,103124
3,CHELSA,1.605645,4.0,0.934264,0.190970,1.101176,103088
4,IS_K,0.191530,1.0,0.963854,0.128994,0.941511,103124
5,DS,0.373193,2.0,0.957460,0.137763,0.941253,103124
